# Metrocuadrado frente a FincaRaíz: ¿cuánto se traslapan? (E8)

**Pregunta:** si sumamos Metrocuadrado al barrido, ¿cuánta oferta nueva aporta y cuánto es el mismo
inmueble publicado en los dos portales? ¿Se puede deduplicar de forma confiable?

**Datos (27-oct-2026, Cali, vivienda: apartamentos y casas):**
- **Metrocuadrado:** barrido de exploración con `src/metrocuadrado_http.py`, solo páginas públicas de listado.
  Cada URL entrega solo su primera página (≤ 50 avisos), pero el filtro público `/{n}-millones/` ordena de
  mayor a menor precio: el barrido **camina por precio** (cada página trae los 50 más caros bajo el techo y
  el siguiente techo es el precio más bajo visto) y parte por habitaciones, baños y estrato los escalones
  donde más de 50 avisos empatan. Cubre el **~99%** de lo que declara el portal (un primer intento por
  barrios cubría el 67%).
- **FincaRaíz:** los avisos activos de `estado/avisos` (barrido nacional).

**Método** (`src/deduplicar.py`): para cada aviso de Metrocuadrado, candidatos de FincaRaíz del mismo tipo a
menos de 500 m, con un puntaje que combina el parecido de la descripción, el área (la mejor combinación de
área total y construida), habitaciones y baños, la distancia y el precio. Un **piso distinto** resta fuerte.
Luego se empareja uno a uno, el par más parecido primero.

Solo cifras agregadas: los avisos concretos no se publican aquí.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE))
from src.deduplicar import emparejar, puntaje
pct = lambda x: f"{x:.0%}"
mil = lambda x: f"{x:,.0f}".replace(",", ".")

m2 = pd.read_parquet(BASE / "data/m2/cali_2026-10-27.parquet")
m2 = m2[m2["tipo"].isin(["Apartamento", "Casa"])].reset_index(drop=True)
seg = pd.read_csv(BASE / "data/m2/cali_2026-10-27_segmentos.csv")
fr = pd.read_parquet(BASE / "data/estado/avisos.parquet")
fr = fr[(fr["ciudad"] == "Cali") & (fr["estado_aviso"] == "activo") & fr["tipo"].isin(["Apartamento", "Apartaestudio", "Casa"])].copy()
fr["tipo"] = fr["tipo"].replace({"Apartaestudio": "Apartamento"})      # Metrocuadrado no separa apartaestudios
fr = fr.reset_index(drop=True)
P = pd.read_parquet(BASE / "data/m2/pares_cali.parquet")
P["puntaje"] = puntaje(P)
declarado = seg[seg["nivel"] == "tipo"].set_index("ruta")["total"]
pd.DataFrame({
    "declarado por Metrocuadrado": [declarado.get("/apartamentos/venta/cali/"), declarado.get("/casas/venta/cali/")],
    "leídos": [(m2.tipo == "Apartamento").sum(), (m2.tipo == "Casa").sum()],
    "FincaRaíz (activos)": [(fr.tipo == "Apartamento").sum(), (fr.tipo == "Casa").sum()],
}, index=["Apartamentos", "Casas"]).assign(**{"cobertura": lambda d: (d.iloc[:, 1] / d.iloc[:, 0]).map(lambda v: f"{v:.1%}")})

,declarado por Metrocuadrado,leídos,FincaRaíz (activos),cobertura
Apartamentos,6060.0,5989,6612,98.8%
Casas,2787.0,2783,3295,99.9%


## 1. Cómo se ve el barrido de Metrocuadrado

Páginas por nivel: `tipo` es la primera página de cada tipo (trae también los avisos sin precio), `precio` los
pasos de la caminata y `empate` las páginas para partir escalones con más de 50 avisos al mismo precio.

In [2]:
print(f"{mil(len(seg))} páginas · {mil(len(m2))} avisos únicos · campos: coordenadas {pct(m2.latitud.notna().mean())}, "
      f"estrato {pct(m2.estrato.notna().mean())}, piso {pct(m2.piso.notna().mean())}, área de lote "
      f"{pct(m2.m2_terreno.notna().mean())}, descripción {pct(m2.descripcion.fillna('').str.len().gt(30).mean())}")
seg.groupby("nivel").agg(paginas=("ruta", "size"), avisos_leidos=("leidos", "sum"))

496 páginas · 8.772 avisos únicos · campos: coordenadas 100%, estrato 100%, piso 64%, área de lote 14%, descripción 100%


,paginas,avisos_leidos
nivel,,
empate,291,9547
precio,203,10026
tipo,2,122


## 2. Revisión manual de pares

Antes de fijar el umbral se revisaron a mano pares de cada franja de puntaje (27-oct). Lo aprendido, ya
incorporado al puntaje:

| Franja | Qué se vio | Ajuste |
|---|---|---|
| ≥ 0,75 | Prácticamente siempre el mismo inmueble (mismo texto; a veces el precio mal digitado en un portal) | — |
| 0,65–0,75 | Casi siempre el mismo: misma área, piso y precio aunque el texto sea de otra inmobiliaria | — |
| 0,55–0,65 | Mezcla: muchos verdaderos (mismo edificio y piso, otro precio) y unidades distintas de un mismo proyecto | Zona a revisar |
| (antes del ajuste) 0,7–0,8 | La misma inmobiliaria usa el **mismo texto para varios apartamentos** de un edificio; solo el **piso** los distingue | Piso distinto resta fuerte |
| (antes del ajuste) 0,8–0,9 | Mismo inmueble con **áreas distintas** (un portal publica la construida, el otro la privada o la del lote) | Área: mejor combinación total/construida |

Es una revisión de muestras, no una medición de precisión: falta un conjunto etiquetado (ver conclusiones).

In [3]:
mejor = P.sort_values("puntaje", ascending=False).drop_duplicates("i")
pd.cut(mejor["puntaje"], [0, .4, .5, .55, .65, .75, .9, 2], right=False).value_counts().sort_index() \
  .rename("avisos de Metrocuadrado según el puntaje de su mejor candidato").to_frame()

,avisos de Metrocuadrado según el puntaje de su mejor candidato
puntaje,
"[0.0, 0.4)",1777
"[0.4, 0.5)",1154
"[0.5, 0.55)",492
"[0.55, 0.65)",766
"[0.65, 0.75)",312
"[0.75, 0.9)",493
"[0.9, 2.0)",2982


## 3. ¿Cuánto se traslapan?

In [4]:
filas = []
for u in (0.55, 0.65, 0.75):
    E = emparejar(P, umbral=u)
    t = len(E) / len(m2)
    solo_m2 = declarado.sum() * (1 - t)
    filas.append({"umbral": u, "pares": len(E), "de Metrocuadrado también en FincaRaíz": t,
                  "solo en Metrocuadrado (estimado)": solo_m2, "oferta única estimada": len(fr) + solo_m2,
                  "aumento frente a solo FincaRaíz": solo_m2 / len(fr)})
pd.DataFrame(filas).set_index("umbral").style.format({
    "pares": mil, "de Metrocuadrado también en FincaRaíz": pct, "solo en Metrocuadrado (estimado)": mil,
    "oferta única estimada": mil, "aumento frente a solo FincaRaíz": lambda v: f"+{v:.0%}"})

,pares,de Metrocuadrado también en FincaRaíz,solo en Metrocuadrado (estimado),oferta única estimada,aumento frente a solo FincaRaíz
umbral,,,,,
0.550000,4.074,46%,4.738,14.645,+48%
0.650000,3.601,41%,5.215,15.122,+53%
0.750000,3.386,39%,5.432,15.339,+55%


In [5]:
E = emparejar(P, umbral=0.65)
ta = m2["tipo_anunciante"].fillna("sin dato")
pd.DataFrame({"avisos": ta.value_counts(), "también en FincaRaíz": ta.groupby(ta).apply(lambda g: g.index.isin(E["i"]).mean())}) \
  .style.format({"avisos": mil, "también en FincaRaíz": pct}).set_caption("Por tipo de anunciante (umbral 0,65)")

,avisos,también en FincaRaíz
tipo_anunciante,,
Constructora,57,5%
Inmobiliaria,8.545,42%
Persona natural,170,9%


## 4. Precios del mismo inmueble en los dos portales

In [6]:
pa, pb = m2["precio"].to_numpy()[E["i"]], fr["precio"].to_numpy()[E["j"]]
ok = (pa > 0) & (pb > 0)                       # precio 0 = sin dato
dif = pa[ok] / pb[ok] - 1
pd.Series({
    "pares con precio en los dos portales": mil(ok.sum()),
    "precio idéntico": pct(np.mean(np.abs(dif) < 0.005)),
    "difieren menos de 5%": pct(np.mean((np.abs(dif) >= 0.005) & (np.abs(dif) < 0.05))),
    "difieren 5% o más": pct(np.mean(np.abs(dif) >= 0.05)),
    "más caro en Metrocuadrado": pct(np.mean(dif > 0.005)),
    "más caro en FincaRaíz": pct(np.mean(dif < -0.005)),
}).to_frame("valor")

,valor
pares con precio en los dos portales,3.600
precio idéntico,96%
difieren menos de 5%,2%
difieren 5% o más,2%
más caro en Metrocuadrado,2%
más caro en FincaRaíz,2%


## 5. Duplicados dentro de los portales

In [7]:
fuertes = P[P["puntaje"] >= 0.65]
multi = fuertes.groupby("j")["i"].nunique()
dup_m2 = m2.duplicated(["tipo", "m2", "habitaciones", "banos", "precio", "barrio"], keep=False).mean()
pd.Series({
    "avisos de FincaRaíz con 2+ avisos de Metrocuadrado muy parecidos (varias inmobiliarias)": mil((multi >= 2).sum()),
    "duplicados exactos dentro de Metrocuadrado (tipo, área, hab., baños, precio, barrio)": f"{dup_m2:.1%}",
}).to_frame("valor")

,valor
avisos de FincaRaíz con 2+ avisos de Metrocuadrado muy parecidos (varias inmobiliarias),703
"duplicados exactos dentro de Metrocuadrado (tipo, área, hab., baños, precio, barrio)",9.9%


## 6. Conclusiones

1. **Metrocuadrado aporta mucha oferta nueva.** Solo el ~41% (39–46% según el umbral) de sus avisos de
   Cali también está en FincaRaíz: sumarlo aumentaría la oferta de vivienda de Cali en **~50%**
   (de ~9.900 a ~15.000 avisos, antes de descontar los duplicados internos de cada portal). Más
   comparables, mejor modelo y mejor alerta.
2. **Cuando el inmueble está en los dos, casi siempre con el mismo precio** (96%): suele ser la misma
   inmobiliaria publicando en ambos. Las diferencias son poco frecuentes pero útiles: margen de
   negociación o errores de digitación.
3. **Duplicar no es solo entre portales:** la misma casa la publican varias inmobiliarias (~700 avisos de
   FincaRaíz con 2+ gemelos en Metrocuadrado) y hay ~10% de duplicados exactos dentro de Metrocuadrado.
4. **La deduplicación es viable con reglas simples** (texto + ficha + distancia + precio, piso como veto).
   Con umbral 0,65 los pares revisados son casi siempre el mismo inmueble.
5. **La cobertura de Metrocuadrado está resuelta** (~99%) con la caminata por precio, en menos páginas
   (~500) que el intento por barrios (682 páginas, 67%). Lo que falta son avisos sin precio y unos pocos
   escalones de empate que ni habitaciones + baños + estrato logran partir.

**Cómo resolverlo (siguientes pasos de E8):**
1. **Conjunto etiquetado:** ~150 pares revisados a mano (los agentes del piloto pueden ayudar) para medir
   precisión y exhaustividad y fijar el umbral con números, no a ojo.
2. **Agrupar, no solo emparejar:** un mismo inmueble puede tener N avisos (portales e inmobiliarias). Pasar
   de pares uno a uno a **grupos** (componentes conectados de los pares fuertes) con un `id_inmueble_unificado`,
   un precio de referencia (el más bajo, el más reciente) y la lista de avisos.
3. **Duplicados dentro de cada portal** con el mismo emparejador (portal contra sí mismo).
4. **Barrido nacional** con la caminata por precio (por ciudad o departamento, para que los empates en precios
   redondos sean manejables), raíz propia (`estado_m2/`) y la vista unificada para la alerta, el modelo y el
   mapa. Con ~99% de cobertura, las bajas pueden usar la misma regla que FincaRaíz (3 ausencias).